In [ ]:
# Cell 1: Imports & 환경변수 기반 OpenAI API 키 설정 + YAML/TXT 디렉터리 로드
import os
import yaml
import json
from datetime import datetime
import openai
from dotenv import load_dotenv
from pathlib import Path
import tiktoken
import time
from openai import OpenAI

# 1) 현재 작업 디렉터리
cwd = Path().cwd()

# 2) .env 파일명(혹은 경로)을 ENV_FILE 환경변수에서 가져오고 로드
#    (예: ENV_FILE=custom.env 으로 지정 가능, 없으면 기본 ".env" 사용)
env_filename = os.getenv("ENV_FILE", ".env")
env_path = cwd / env_filename

print(f"Loading environment variables from {env_path}")
if not env_path.exists():
    raise FileNotFoundError(f"Environment file '{env_filename}' not found at {env_path}")

load_dotenv(dotenv_path=env_path)

# 3) OPENAI_API_KEY 로드
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise EnvironmentError("환경변수 OPENAI_API_KEY가 설정되지 않았습니다.")
openai.api_key = api_key
print("OPENAI_API_KEY loaded successfully")

# 4) YAML 디렉터리 및 TXT 디렉터리 환경변수 로드
#    (.env에 PROMPT_YAML_DIR, TARGET_TXT_DIR를 미리 정의해야 함)
YAML_DIR = os.getenv("PROMPT_YAML_DIR", ".")
TXT_DIR  = os.getenv("TARGET_TXT_DIR", ".")

# 5) Path 객체로 변환 및 유효성 검사
yaml_dir_path = Path(YAML_DIR)
txt_dir_path  = Path(TXT_DIR)

if not yaml_dir_path.exists() or not yaml_dir_path.is_dir():
    raise FileNotFoundError(f"YAML 디렉터리를 찾을 수 없습니다: {yaml_dir_path}")

if not txt_dir_path.exists() or not txt_dir_path.is_dir():
    raise FileNotFoundError(f"TXT 디렉터리를 찾을 수 없습니다: {txt_dir_path}")

print(f"YAML_DIR: {yaml_dir_path}")
print(f"TXT_DIR: {txt_dir_path}")

In [ ]:
# Cell 2: PromptManager 정의
class PromptManager:
    def __init__(self, yaml_path: str):
        if not os.path.isfile(yaml_path):
            raise FileNotFoundError(f"프롬프트 YAML을 찾을 수 없습니다: {yaml_path}")
        with open(yaml_path, 'r', encoding='utf-8') as f:
            data = yaml.safe_load(f)
        self.prompts = data.get('prompts', {})

    def list_versions(self, product: str) -> list[str]:
        """해당 상품에 사용 가능한 버전 목록 반환"""
        return [e['version'] for e in self.prompts.get(product, [])]

    def get_prompt(self, product: str, version: str = None) -> str:
        """상품(product)과 버전(version)에 맞는 template 반환 (버전 누락 시 최신)"""
        entries = self.prompts.get(product)
        if not entries:
            raise KeyError(f"상품 정의 없음: {product}")
        if version:
            for e in entries:
                if e['version'] == version:
                    return e['template']
            raise KeyError(f"{product}에 버전 {version} 없음")
        # 최신 버전 (버전 문자열 내림차순 가정)
        latest = sorted(entries, key=lambda e: e['version'], reverse=True)[0]
        return latest['template']

In [ ]:
# Cell 3: 텍스트 로더, 청크 분할, LLM 호출/파싱
def load_text(file_path: str) -> str:
    with open(file_path, 'r', encoding='utf-8') as f:
        return f.read()

def split_text(text: str, max_chars: int = 12000) -> list[str]:
    """
    max_chars 내외로 청크를 나눕니다.
    문단 단위(\n)로 쪼갠 뒤 합치는 방식으로 크게 깨지지 않게 분할.
    """
    paras = text.split('\n')
    chunks, cur = [], ""
    for p in paras:
        if len(cur) + len(p) + 1 <= max_chars:
            cur += p + "\n"
        else:
            chunks.append(cur)
            cur = p + "\n"
    if cur:
        chunks.append(cur)
    return chunks

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def call_llm(prompt: str, model: str = 'gpt-4.1-mini', max_retries: int = 3, seed: int = 43, 
             temperature: float = 0.0, top_p: float = 0.05, 
             frequency_penalty: float = 0.0, presence_penalty: float = 0.0):
    for attempt in range(1, max_retries+1):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=[{"role":"user","content":prompt}],
                temperature=temperature,   
                top_p=top_p,             
                frequency_penalty=frequency_penalty,  
                presence_penalty=presence_penalty, 
                seed=seed
            )
            return resp.choices[0].message.content
        except Exception as e:
            if attempt == max_retries:
                raise
            time.sleep(2 ** (attempt-1))
            
def parse_llm_response(text: str) -> dict:
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start, end = text.find('{'), text.rfind('}') + 1
        return json.loads(text[start:end])

In [ ]:
# Cell 4: JSON 저장 및 청크 기반 처리 함수 (상품 구분 로직 제거)

def save_json_output(data: dict, output_path: str):
    os.makedirs(os.path.dirname(output_path) or '.', exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

# Cell 5: JSON 저장 및 청크 토큰 카운트 유지 + 전체 문서 단일 결과 처리
def process_file(txt_path: str, output_dir: str, pm: PromptManager, version: str, model: str, seed: int = 42):
    print(f"▶ Processing TXT: {os.path.basename(txt_path)}")
    # 1) 원본 텍스트 읽기
    raw_text = load_text(txt_path)
    # 2) 프롬프트 템플릿 불러오기
    template = pm.get_prompt(version)

    # 3) 텍스트 청크 분할 & 토큰 수 계산
    chunks = split_text(raw_text)
    try:
        enc = tiktoken.encoding_for_model(model)
    except KeyError:
        enc = tiktoken.get_encoding("cl100k_base")

    for i, chunk in enumerate(chunks, 1):
        # 청크 단위로 prompt 채운 뒤 토큰 수 계산
        prompt_chunk = template.replace("{{text}}", chunk)
        token_count  = len(enc.encode(prompt_chunk))
        print(f"[Chunk {i}] 토큰 수: {token_count}")

    # 4) 전체 텍스트 한 번만 LLM 호출 → JSON 파싱
    prompt_full = template.replace("{{text}}", raw_text)
    resp_text   = call_llm(prompt_full, model, seed=seed)
    parsed      = parse_llm_response(resp_text)

    # 5) 결과 저장 (파일당 하나의 JSON)
    ts       = datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')
    safe_ver = version.replace('.', '_')
    base     = os.path.splitext(os.path.basename(txt_path))[0]
    out_name = f"{base}_v{safe_ver}_{ts}.json"
    os.makedirs(output_dir, exist_ok=True)
    out_path = os.path.join(output_dir, out_name)

    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(parsed, f, ensure_ascii=False, indent=4)

    print(f"{txt_path} → {out_name} (전체 문서 단일 JSON) 완료")

In [ ]:
# Cell 6: 실행 예시 (환경변수로 설정된 디렉터리에서 자동 탐색) - 숫자 버전 기반 최신 YAML 선택

import re

# 0) 처리할 상품 타입 설정 (FRN 또는 bond_forward)
PRODUCT_TYPE = "FRN"  # "FRN" 또는 "bond_forward" 선택

# 상품 타입별 설정
if PRODUCT_TYPE == "FRN":
    yaml_pattern = "prompts_FRN*.yaml"
    yaml_base_name = "prompts_FRN.yaml"
    yaml_regex = r'prompts_FRN(\d+)\.yaml'
    output_dir = "../dataset/처리/FRN/ocr_json"
    version_key = "v_frn_feed"
elif PRODUCT_TYPE == "선도채권":
    yaml_pattern = "prompts_bond_forward*.yaml"
    yaml_base_name = "prompts_bond_forward.yaml"
    yaml_regex = r'prompts_bond_forward.*?(\d+)\.yaml'
    output_dir = "../dataset/처리/채권선도/ocr_json"
    version_key = "bond_forward"
    # 선도채권의 경우 yaml_선도채권 폴더 사용
    yaml_dir_path = Path("yaml_선도채권")
else:
    raise ValueError(f"지원하지 않는 상품 타입: {PRODUCT_TYPE}")

print(f"처리 대상 상품: {PRODUCT_TYPE}")
print(f"YAML 디렉터리: {yaml_dir_path}")

# 1) YAML 파일 자동 탐색
yamls = list(yaml_dir_path.glob(yaml_pattern))
if not yamls:
    raise FileNotFoundError(f"{yaml_pattern} 파일이 하나도 없습니다: {yaml_dir_path}")

# 2) 숫자 버전 추출 및 정렬
def extract_version_number(yaml_path, product_type):
    """파일명에서 숫자 버전을 추출합니다."""
    filename = yaml_path.name
    
    if product_type == "FRN":
        # prompts_FRN숫자.yaml 패턴 매칭
        match = re.search(r'prompts_FRN(\d+)\.yaml', filename)
        if match:
            return int(match.group(1))
        elif filename == "prompts_FRN.yaml":
            return 0
    elif product_type == "선도채권":
        # prompts_bond_forward숫자.yaml 패턴 매칭 (예: prompts_bond_forward2.yaml -> 2)
        match = re.search(r'prompts_bond_forward(\d+)\.yaml', filename)
        if match:
            return int(match.group(1))
        elif filename == "prompts_bond_forward.yaml":
            return 0
    
    return -1  # 패턴에 맞지 않는 파일

# 3) 버전 번호 기준으로 정렬하여 가장 높은 버전 선택
yamls_with_versions = [(yaml_path, extract_version_number(yaml_path, PRODUCT_TYPE)) for yaml_path in yamls]
# 유효한 버전 번호를 가진 파일들만 필터링 (버전 번호가 -1이 아닌 것들)
valid_yamls = [(path, ver) for path, ver in yamls_with_versions if ver >= 0]

if not valid_yamls:
    raise FileNotFoundError(f"유효한 {yaml_pattern} 파일이 없습니다: {yaml_dir_path}")

# 버전 번호 기준으로 내림차순 정렬하여 최신 버전 선택
latest_yaml_path, latest_version = sorted(valid_yamls, key=lambda x: x[1], reverse=True)[0]
PROMPT_YAML = str(latest_yaml_path)

print(f"발견된 {yaml_pattern} 파일들:")
for path, ver in sorted(valid_yamls, key=lambda x: x[1], reverse=True):
    marker = " ← 선택됨" if path == latest_yaml_path else ""
    print(f" - {path.name} (버전: {ver}){marker}")

print(f"\n사용할 YAML: {PROMPT_YAML}")

# 4) TXT 파일 자동 탐색: txt_dir_path 밑의 모든 *.txt
txts = list(txt_dir_path.glob("*.txt"))
if not txts:
    raise FileNotFoundError(f"TXT 파일이 하나도 없습니다: {txt_dir_path}")
print("처리할 TXT 파일 목록:")
for t in txts:
    print(" -", t.name)

# 5) PromptManager 초기화 (최신 버전 YAML 사용)
pm = PromptManager(PROMPT_YAML)

# 6) 출력 디렉터리, 모델, 버전 설정
OUTPUT_DIR = output_dir
MODEL_NAME = "gpt-4.1-mini"
VERSION = version_key

print(f"출력 디렉터리: {OUTPUT_DIR}")
print(f"사용할 버전: {VERSION}")

# 7) 모든 TXT 파일을 순차적으로 처리
for txt_path in txts:
    process_file(str(txt_path), OUTPUT_DIR, pm, VERSION, MODEL_NAME)
    print(f"{txt_path.name} 처리 완료")